In [1]:
from sklearn.base import clone
import pandas as pd
from mibici_mlops.modeling.train_p3 import P3Trainer
from mibici_mlops.config import P3_PERFILES_PATH, ESTACIONES_PATH, RANDOM_STATE

2026-10-25 14:34:51.505 | INFO     | mibici_mlops.config:<module>:16 - PROJ_ROOT path is: /Users/timedina/Documents/timj/Docs/Estudios/MNA/MLOps/Activities/05 - MiBici + MLFlow/code/MiBici_MLOps
/Users/timedina/Documents/timj/Docs/Estudios/MNA/MLOps/Activities/05 - MiBici + MLFlow/code/MiBici_MLOps/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
perfiles = pd.read_parquet(P3_PERFILES_PATH)
estaciones = pd.read_csv(ESTACIONES_PATH)

## K-Means

In [3]:
from sklearn.cluster import KMeans

res = []
for n_clusters in [2, 3, 4, 5]:
    for n_init in [10, 20, 30]:
        parametros = {
            'n_clusters': n_clusters,
            'n_init': n_init,
            'random_state': RANDOM_STATE
        }
        modelo = KMeans(**parametros)
        p3train = P3Trainer(perfiles, estaciones, modelo)
        p3train.fit_predict()
        res.append({**parametros, **p3train.metricas})
resdf = pd.DataFrame(res)
resdf.sort_values(
    by=["ari_estabilidad", "val_silhouette", "n_clusters", "n_init"], 
    ascending=[False, False, True, True]
)


,n_clusters,n_init,random_state,train_silhouette,ari_estabilidad,val_silhouette,inertia
0,2,10,67,0.277634,0.752973,0.229486,2.086036
1,2,20,67,0.277634,0.752973,0.229486,2.086036
2,2,30,67,0.277634,0.752973,0.229486,2.086036
3,3,10,67,0.303045,0.748535,0.260651,1.623822
4,3,20,67,0.303045,0.748535,0.260651,1.623822
5,3,30,67,0.302750,0.748174,0.260273,1.623810
6,4,10,67,0.214343,0.581363,0.177368,1.412482
8,4,30,67,0.212157,0.549766,0.175852,1.409879
7,4,20,67,0.210297,0.549370,0.175009,1.410477
10,5,20,67,0.190710,0.521231,0.149692,1.277598


La configuración que seguirá siendo el baseline es `n_clusters=3` y `n_init=10`, aunque `n_clusters=2` tiene un mejor ARI, es muy poca la diferencia y `3` nos da mejor silhouette tanto en entrenamiento como en validación.

## Gaussian Mixture

In [4]:
from sklearn.mixture import GaussianMixture

res = []
for n_components in [2, 3, 4, 5]:
    for n_init in [1, 5, 10]:
        parametros = {
            "n_components": n_components,
            "n_init": n_init,
            "random_state": RANDOM_STATE,
            "covariance_type": "diag",
        }
        modelo = GaussianMixture(**parametros)
        p3train = P3Trainer(perfiles, estaciones, modelo)
        p3train.fit_predict()
        res.append({**parametros, **p3train.metricas})
resdf = pd.DataFrame(res)
resdf.sort_values(
    by=["ari_estabilidad", "val_silhouette", "n_components", "n_init"], 
    ascending=[False, False, True, True]
)

,n_components,n_init,random_state,covariance_type,train_silhouette,ari_estabilidad,val_silhouette,bic
1,2,5,67,diag,0.264716,0.705900,0.217710,-149928.472818
2,2,10,67,diag,0.264716,0.705900,0.217710,-149928.472818
0,2,1,67,diag,0.265484,0.687495,0.217710,-149928.505417
3,3,1,67,diag,0.247949,0.649079,0.214432,-152099.867586
10,5,5,67,diag,0.158191,0.648868,0.097289,-154331.895389
4,3,5,67,diag,0.156494,0.610943,0.090283,-152150.328720
5,3,10,67,diag,0.156494,0.610943,0.090283,-152150.328720
7,4,5,67,diag,0.144373,0.607093,0.092121,-153630.606751
8,4,10,67,diag,0.144373,0.607093,0.092121,-153630.758982
6,4,1,67,diag,0.145553,0.601929,0.092121,-153639.255319


De estos resultados, `n_components=2` nos dá el mejor ARI y `n_init=5` nos da la misma silhouette y bic que 10 pero con menor complejidad.

El ARI de KMeans(3,10) supera a Gaussian Mixture.

## Agglomerative Clustering

In [7]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import adjusted_rand_score, silhouette_score

columnas = [
    c for c in perfiles.columns
    if c.startswith(("sal_", "lle_"))
]

train = (
    perfiles.loc[perfiles["conjunto"].eq("entrenamiento")]
    .set_index("estacion")
)
val = (
    perfiles.loc[perfiles["conjunto"].eq("validacion")]
    .set_index("estacion")
)

X_train = train[columnas].astype("float64")
X_val = val[columnas].astype("float64")

resultados = []

for n_clusters in [2, 3, 4, 5]:
    for linkage in ["ward", "complete", "average", "single"]:
        parametros = {
            "n_clusters": n_clusters,
            "linkage": linkage,
            "metric": "euclidean",
        }

        modelo_train = AgglomerativeClustering(**parametros)
        etiquetas_train = modelo_train.fit_predict(X_train)

        modelo_val = AgglomerativeClustering(**parametros)
        etiquetas_val = modelo_val.fit_predict(X_val)

        grupos_train = pd.Series(etiquetas_train, index=X_train.index)
        grupos_val = pd.Series(etiquetas_val, index=X_val.index)
        comunes = grupos_train.index.intersection(grupos_val.index)

        resultados.append({
            **parametros,
            "train_silhouette": silhouette_score(X_train, etiquetas_train),
            "val_silhouette": silhouette_score(X_val, etiquetas_val),
            "ari_estabilidad": adjusted_rand_score(
                grupos_train.loc[comunes],
                grupos_val.loc[comunes],
            ),
        })

resdf = pd.DataFrame(resultados)
resdf.sort_values(
    by=["ari_estabilidad", "val_silhouette", "n_clusters", "linkage"],
    ascending=[False, False, True, True],
)

,n_clusters,linkage,metric,train_silhouette,val_silhouette,ari_estabilidad
10,4,average,euclidean,0.237897,0.369088,0.746508
14,5,average,euclidean,0.191529,0.360022,0.728885
1,2,complete,euclidean,0.474479,0.457972,0.713420
2,2,average,euclidean,0.613126,0.657999,0.660483
4,3,ward,euclidean,0.291133,0.251996,0.578314
11,4,single,euclidean,0.226034,0.350403,0.492357
15,5,single,euclidean,0.222865,0.311941,0.435515
0,2,ward,euclidean,0.250893,0.227842,0.418733
8,4,ward,euclidean,0.171194,0.255189,0.383020
13,5,complete,euclidean,0.183516,0.183106,0.370850


- Más estable: `4, average`: ARI 0.747 y silhouette de validación 0.369.
- Grupos más separados: `2, average`: silhouette de validación 0.658 pero ARI 0.660
- Punto medio: `2, complete`: ARI 0.713 y silhouette de validación 0.458.

Debido a lo ya identificado (p.e. que uno de los grupos está relacionado a la vía recreativa), no nos importa que los grupos estén más separados, por lo que, para Agglomerative Clustering, el mejor modelo sería problablemente `n_clusters=4` y `linkeage=average`, pero con un ARI muy similar al de KMeans(3, 10) (e incluso ligeramente menor), seguimos con el modelo baseline.